# PakWheels Car Scraper (Auto-Setup Version)

**How to use this — you don't need to understand the code:**

1. Click **Run All** at the top of VS Code (or run every cell top to bottom, one by one, using the ▶ button on each cell).
2. Wait — it will print progress messages as it works.
3. At the end, a file called `pakwheels_used_cars.csv` will appear in the same folder as this notebook. That's your dataset.
4. If it prints an error or says "0 rows found", just copy the printed messages and send them to me — I'll fix it for you.

In [ ]:
# STEP 1: Install everything needed (only runs once, safe to re-run)
import sys
!{sys.executable} -m pip install requests beautifulsoup4 pandas lxml --quiet
print("Setup complete.")

In [ ]:
# STEP 2: Load libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, random, re
from datetime import datetime

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

BASE_URL = "https://www.pakwheels.com/used-cars/search/-/"

print("Ready.")

In [ ]:
# STEP 3: Auto-detect which selectors work on the live site right now.
# It tries several known patterns and picks whichever one actually finds car listings.

CANDIDATE_SELECTORS = [
    {"card": "li.classified-listing", "title": "h3", "price": ".price-details"},
    {"card": "div.search-vehicle", "title": "h3", "price": ".price-details"},
    {"card": "li[id^='ad_']", "title": "h3", "price": ".price-details"},
    {"card": "div.classified-listing", "title": ".car-name", "price": ".price"},
    {"card": "article", "title": "h3", "price": ".price-details"},
]

def fetch_page_html(page_num=1):
    url = f"{BASE_URL}?page={page_num}"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text

def detect_selectors():
    print("Testing connection to PakWheels...")
    try:
        html = fetch_page_html(1)
    except Exception as e:
        print(f"Could not reach PakWheels at all. Error: {e}")
        print("This usually means your internet/firewall is blocking it, or the site is temporarily down.")
        return None

    soup = BeautifulSoup(html, "lxml")

    for candidate in CANDIDATE_SELECTORS:
        cards = soup.select(candidate["card"])
        if len(cards) >= 5:  # treat 5+ matches as a real listing grid, not a false positive
            print(f"Found working selector: {candidate} -> {len(cards)} listings detected")
            return candidate

    print("None of the built-in selector patterns matched.")
    print("The site's HTML has likely changed. Saving the raw page as 'debug_page.html' —")
    print("send me that file and I will generate the correct selectors for you.")
    with open("debug_page.html", "w", encoding="utf-8") as f:
        f.write(html)
    return None

WORKING_SELECTORS = detect_selectors()

In [ ]:
# STEP 4: Helper functions to clean price and title text into structured fields

def parse_price(text):
    if not text:
        return None
    t = text.strip().lower().replace("pkr", "").strip()
    m = re.search(r"[\d,.]+", t)
    if not m:
        return None
    num = float(m.group().replace(",", ""))
    if "crore" in t:
        return int(num * 10_000_000)
    if "lac" in t or "lakh" in t:
        return int(num * 100_000)
    return int(num)

def parse_title(title):
    year_match = re.search(r"(19|20)\d{2}", title)
    year = int(year_match.group()) if year_match else None
    words = title.replace(str(year), "").split() if year else title.split()
    brand = words[0] if len(words) > 0 else None
    model = words[1] if len(words) > 1 else None
    variant = " ".join(words[2:]) if len(words) > 2 else None
    return brand, model, variant, year

print("Helper functions ready.")

In [ ]:
# STEP 5: The actual scraping loop

NUM_PAGES = 5   # increase this later once you confirm it's working, e.g. to 50 or 100
SLEEP_MIN, SLEEP_MAX = 3, 6  # polite delay between pages, in seconds

all_rows = []

if WORKING_SELECTORS is None:
    print("Skipping scrape — no working selector was detected in Step 3. See the message above.")
else:
    for page in range(1, NUM_PAGES + 1):
        try:
            html = fetch_page_html(page)
        except Exception as e:
            print(f"Page {page}: failed to load ({e}), skipping")
            continue

        soup = BeautifulSoup(html, "lxml")
        cards = soup.select(WORKING_SELECTORS["card"])
        print(f"Page {page}: {len(cards)} listings found")

        for card in cards:
            try:
                title_el = card.select_one(WORKING_SELECTORS["title"])
                price_el = card.select_one(WORKING_SELECTORS["price"])
                link_el = card.select_one("a")

                title = title_el.get_text(strip=True) if title_el else None
                price_raw = price_el.get_text(strip=True) if price_el else None
                link = link_el["href"] if link_el and link_el.has_attr("href") else None

                if not title:
                    continue

                brand, model, variant, year = parse_title(title)

                # grab any list items in the card as raw specs text (mileage, fuel, transmission etc.)
                spec_items = [li.get_text(strip=True) for li in card.select("li")]
                specs_raw = " | ".join(spec_items)

                all_rows.append({
                    "title": title,
                    "brand": brand,
                    "model": model,
                    "variant": variant,
                    "year": year,
                    "price_pkr": parse_price(price_raw),
                    "price_raw": price_raw,
                    "specs_raw": specs_raw,
                    "link": link,
                    "scraped_at": datetime.now().strftime("%Y-%m-%d")
                })
            except Exception as e:
                continue

        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

    print(f"\nTotal rows collected: {len(all_rows)}")

In [ ]:
# STEP 6: Clean up and extract mileage / fuel type / transmission / engine size from specs_raw

df = pd.DataFrame(all_rows)

if len(df) == 0:
    print("No data was scraped, so there's nothing to clean. Check the messages in Step 3 and Step 5 above.")
else:
    df = df.drop_duplicates(subset=["title", "price_raw"])
    df = df.dropna(subset=["price_pkr"])

    def extract_spec(specs_raw, pattern):
        if not isinstance(specs_raw, str):
            return None
        for part in specs_raw.split("|"):
            part = part.strip()
            if re.search(pattern, part, re.IGNORECASE):
                return part
        return None

    df["mileage_raw"] = df["specs_raw"].apply(lambda x: extract_spec(x, r"km"))
    df["fuel_type"] = df["specs_raw"].apply(lambda x: extract_spec(x, r"petrol|diesel|hybrid|cng|electric"))
    df["transmission"] = df["specs_raw"].apply(lambda x: extract_spec(x, r"automatic|manual"))
    df["engine_cc_raw"] = df["specs_raw"].apply(lambda x: extract_spec(x, r"cc\b"))

    df["mileage_km"] = df["mileage_raw"].str.extract(r"([\d,]+)")[0].str.replace(",", "", regex=False)
    df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")

    df["engine_cc"] = df["engine_cc_raw"].str.extract(r"([\d,]+)")[0].str.replace(",", "", regex=False)
    df["engine_cc"] = pd.to_numeric(df["engine_cc"], errors="coerce")

    print(f"Cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    display(df.head(10))

In [ ]:
# STEP 7: Save the final CSV

if len(df) > 0:
    final_cols = ["title", "brand", "model", "variant", "year", "price_pkr",
                  "mileage_km", "fuel_type", "transmission", "engine_cc", "link", "scraped_at"]
    final_df = df[final_cols].reset_index(drop=True)
    final_df.to_csv("pakwheels_used_cars.csv", index=False)
    print(f"Saved {len(final_df)} rows to pakwheels_used_cars.csv")
    print("This file is in the same folder as this notebook. You can now open it in Excel or upload it to Kaggle.")
else:
    print("Nothing to save yet — the scrape didn't collect any rows.")

## If it didn't work

If Step 3 printed **"None of the built-in selector patterns matched"**, a file called `debug_page.html` will be saved next to this notebook. Send me that file (or just open it and copy-paste a chunk of the HTML around one car listing) and I'll write the exact selectors for you — no guesswork needed at that point.